# Data Cleaning Pipeline

Runs the cleaning pipeline defined in `src/clean.py` on the Jigsaw raw CSVs and writes outputs to `dataset/processed/`.

- **Train**: full cleaning (universal + classical), drop non-English, empty, duplicates.
- **Test**: same text cleaning, but **no row drops** (preserve all 153k rows for inference).

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import csv
import json
import pandas as pd

from src.clean import clean_dataframe, clean_text_universal
from src.setup_nlp import ensure_nlp_assets

ensure_nlp_assets(verbose=True)

RAW_TRAIN = PROJECT_ROOT / 'dataset' / 'raw' / 'train' / 'train.csv'
RAW_TEST  = PROJECT_ROOT / 'dataset' / 'raw' / 'test'  / 'test.csv'
OUT_DIR   = PROJECT_ROOT / 'dataset' / 'processed'
OUT_DIR.mkdir(parents=True, exist_ok=True)

LABEL_COLS = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

## 1. Clean the training set

In [ ]:
raw_train = pd.read_csv(RAW_TRAIN)
print(f'Raw train rows: {len(raw_train):,}')

label_dist_before = {c: raw_train[c].mean() for c in LABEL_COLS}

train_clean, train_stats = clean_dataframe(
    raw_train,
    text_col='comment_text',
    classical=True,
    drop_lang=True,
    drop_empty=True,
    drop_dups=True,
)

label_dist_after = {c: train_clean[c].mean() for c in LABEL_COLS}

out_train = OUT_DIR / 'train_clean.csv'
train_clean.to_csv(out_train, index=False, quoting=csv.QUOTE_ALL)
print(f'Wrote {out_train} ({len(train_clean):,} rows)')
train_stats

## 2. Clean the test set (preserve all rows)

In [ ]:
raw_test = pd.read_csv(RAW_TEST)
print(f'Raw test rows: {len(raw_test):,}')

test_clean, test_stats = clean_dataframe(
    raw_test,
    text_col='comment_text',
    classical=True,
    drop_lang=False,
    drop_empty=False,
    drop_dups=False,
)

out_test = OUT_DIR / 'test_clean.csv'
test_clean.to_csv(out_test, index=False, quoting=csv.QUOTE_ALL)
print(f'Wrote {out_test} ({len(test_clean):,} rows)')
test_stats

## 3. Verification

In [ ]:
print('Row counts')
print(f'  raw train: {len(raw_train):,}')
print(f'  clean train: {len(train_clean):,}')
print(f'  dropped (lang): {train_stats["n_dropped_lang"]:,}')
print(f'  dropped (empty): {train_stats["n_dropped_empty"]:,}')
print(f'  dropped (dup): {train_stats["n_dropped_dup"]:,}')
print()
print('Label distribution drift (positive rate before -> after)')
for c in LABEL_COLS:
    before = label_dist_before[c]
    after  = label_dist_after[c]
    print(f'  {c:<14} {before:.4f} -> {after:.4f}  (delta {after - before:+.4f})')

In [ ]:
raw_sample = raw_train[['id', 'comment_text']].sample(20, random_state=0).reset_index(drop=True)
raw_sample['cleaned'] = raw_sample['comment_text'].map(clean_text_universal)
raw_sample[['comment_text', 'cleaned']]

In [ ]:
lengths = train_clean['comment_text'].str.len()
print('Cleaned-text length stats:')
print(lengths.describe())

print('\nTop 5 longest cleaned rows:')
print(train_clean.iloc[lengths.nlargest(5).index]['comment_text'].tolist())

print('\nTop 5 shortest cleaned rows:')
print(train_clean.iloc[lengths.nsmallest(5).index]['comment_text'].tolist())

assert lengths.min() > 0, 'empty rows leaked into output'
assert train_clean['comment_text'].notna().all(), 'NaN rows leaked into output'
print('\nAssertions OK')

## 4. Write report

In [ ]:
report = {
    'train': {**train_stats,
              'label_dist_before': {k: round(v, 6) for k, v in label_dist_before.items()},
              'label_dist_after':  {k: round(v, 6) for k, v in label_dist_after.items()}},
    'test':  test_stats,
}
report_path = OUT_DIR / 'cleaning_report.json'
report_path.write_text(json.dumps(report, indent=2))
print(f'Wrote {report_path}')
report